# Hybrid NER Labeling: Electra Model + Regex + Gazetteer

**Dataset:** `raw_data_2_pre.csv` — ~13,900 bài báo tiếng Việt

## Kiến trúc Pipeline

```
Raw Text
    │
    ▼
┌─────────────────────────────────────────────────────────┐
│  Text Normalization                                     │
│  Unicode NFC · NBSP/ZWS → loại · smart quotes → ASCII  │
│  multi-space → gộp                                      │
├─────────────────────────────────────────────────────────┤
│  Layer 1: Electra NER Model (GPU batch)                 │
│  NlpHUST/ner-vietnamese-electra-base  F1=0.92           │
│  → PERSON, ORGANIZATION, LOCATION                       │
├─────────────────────────────────────────────────────────┤
│  Layer 2: Regex Patterns                                │
│  → MONEY, DATE, PERCENT  (structured)                   │
│  → PRODUCT families: iPhone/Galaxy/Pixel/iOS/Android…   │
├─────────────────────────────────────────────────────────┤
│  Layer 3: Gazetteer  (Aho-Corasick O(n))               │
│  → PRODUCT (HARDWARE + SOFTWARE), EVENT, INDUSTRY       │
│  → Patch: ORGANIZATION, LOCATION, PERSON bị model miss  │
│  → INDUSTRY: contextual filter (trigger word ±50 chars) │
└─────────────────────────────────────────────────────────┘
                        ↓
              Merge · Priority: Model > Regex > Gazetteer
              Longest match wins · Overlap resolved
                        ↓
              Canonicalization (alias → canonical)
                        ↓
                  BIO Output / JSON / CoNLL
```

**Entity types:** PERSON · ORGANIZATION · LOCATION · PRODUCT · EVENT · INDUSTRY · MONEY · DATE · PERCENT

In [14]:
import pandas as pd
import re
import json
import time
import unicodedata
import warnings
from collections import Counter
from typing import List, Tuple, Dict, Optional

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    pipeline,
)

try:
    import ahocorasick
    HAS_AHOCORASICK = True
except ImportError:
    HAS_AHOCORASICK = False
    print(" pyahocorasick không được cài. Fallback sang str.find() (chậm hơn ~20x).")
    print("  Cài: pip install pyahocorasick")

warnings.filterwarnings("ignore")

assert torch.cuda.is_available(), (
    "pip install --force-reinstall torch torchvision torchaudio "
    "--index-url https://download.pytorch.org/whl/cu128"
)

DEVICE = 0
gpu_props = torch.cuda.get_device_properties(0)
print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {gpu_props.total_memory / 1024**3:.1f} GB")
print(f"CUDA    : {torch.version.cuda}")

df = pd.read_csv("CSV/raw_data_2_pre.csv")
print(f"\nTổng bài báo: {len(df):,}")
print(f"Cột: {list(df.columns)}")
print(f"Missing:\n{df.isnull().sum()}")
df.head(2)

PyTorch : 2.10.0+cu128
GPU     : NVIDIA GeForce RTX 3070 Laptop GPU
VRAM    : 8.0 GB
CUDA    : 12.8

Tổng bài báo: 13,906
Cột: ['id', 'title', 'content', 'source', 'date']
Missing:
id         0
title      0
content    0
source     0
date       0
dtype: int64


,id,title,content,source,date
0,1,Motorola chuẩn bị ra mắt phiên bản điện thoại ...,Thiết bị mới đã xuất hiện trên nền tảng Geekbe...,vietnamnet,2024-09-24
1,2,"2 tỷ phú quốc tế 'xông đất', hứa đưa Bình Định...","Ngày 16/1, tỉnh Bình Định đã chào đón hai nhà ...",vietnamnet,2025-01-16


## Text Normalization

Chuẩn hoá text trước khi NER: Unicode NFC, whitespace, ký tự đặc biệt → tránh miss match.

In [15]:
_NORMALIZE_MAP = {
    "\u00a0": " ",   # non-breaking space
    "\u200b": "",    # zero-width space
    "\u200c": "",    # zero-width non-joiner
    "\u200d": "",    # zero-width joiner
    "\ufeff": "",    # BOM
    "\u2013": "-",   # en-dash → hyphen
    "\u2014": "-",   # em-dash → hyphen
    "\u2018": "'",   # left single quote
    "\u2019": "'",   # right single quote
    "\u201c": '"',   # left double quote
    "\u201d": '"',   # right double quote
    "\u2026": "...", # ellipsis
    "\u202f": " ",   # narrow no-break space
    "\u2003": " ",   # em space
    "\u2002": " ",   # en space
    "\u2009": " ",   # thin space
    "\u200a": " ",   # hair space
    "\u3000": " ",   # ideographic space (fullwidth)
}
_NORM_RE = re.compile("|".join(re.escape(k) for k in _NORMALIZE_MAP))
_MULTI_SPACE = re.compile(r"[ \t]{2,}")


def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFC", text)
    text = _NORM_RE.sub(lambda m: _NORMALIZE_MAP[m.group()], text)
    text = _MULTI_SPACE.sub(" ", text)
    return text.strip()


test_cases = [
    ("NBSP+em-space+ZWSP", "Ngày\u00a016/1,\u2003ông\u200bRoland"),
    ("en-dash",            "GDP tăng\u2013trưởng 6,5%"),
    ("smart quotes",       "\u201cChuyển đổi số\u201d nâng cao"),
    ("multi-space+NBSP",   "Giá  tương   đương   12  triệu\u00a0đồng"),
    ("NFD decomposed",     "Ha\u0300 No\u0323i"),
]
for label, tc in test_cases:
    norm = normalize_text(tc)
    print(f"  [{label:20s}] '{norm}'")

  [NBSP+em-space+ZWSP  ] 'Ngày 16/1, ôngRoland'
  [en-dash             ] 'GDP tăng-trưởng 6,5%'
  [smart quotes        ] '"Chuyển đổi số" nâng cao'
  [multi-space+NBSP    ] 'Giá tương đương 12 triệu đồng'
  [NFD decomposed      ] 'Hà Nọi'


## Layer 1: Electra NER Model

`NlpHUST/ner-vietnamese-electra-base` — Fine-tuned trên VLSP 2018, F1 = 0.92

Nhận diện: **PERSON** (F1=0.97), **ORGANIZATION** (F1=0.88), **LOCATION** (F1=0.94)

In [16]:
MODEL_NAME = "NlpHUST/ner-vietnamese-electra-base"

print(f"Loading model: {MODEL_NAME} ...")
ner_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
ner_model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)
ner_model = ner_model.to("cuda")
ner_model.eval()

# batch_size=16 tối ưu cho RTX 3070 8GB, model ~110M params chỉ tốn ~0.5GB VRAM
ner_pipeline = pipeline(
    "ner",
    model=ner_model,
    tokenizer=ner_tokenizer,
    aggregation_strategy="simple",
    device=DEVICE,
    batch_size=16,
)

ELECTRA_LABEL_MAP = {
    "PER": "PERSON",
    "PERSON": "PERSON",
    "ORG": "ORGANIZATION",
    "ORGANIZATION": "ORGANIZATION",
    "LOC": "LOCATION",
    "LOCATION": "LOCATION",
    "MISC": "MISCELLANEOUS",
    "MISCELLANEOUS": "MISCELLANEOUS",
}

vram_used = torch.cuda.memory_allocated(0) / 1024**2
print(f"Model loaded on GPU ({vram_used:.0f} MB VRAM)")
print(f"Label map: {ner_model.config.id2label}")
print(f"Batch size: 16")

Loading model: NlpHUST/ner-vietnamese-electra-base ...


Device set to use cuda:0


Model loaded on GPU (1027 MB VRAM)
Label map: {0: 'B-LOCATION', 1: 'B-MISCELLANEOUS', 2: 'B-ORGANIZATION', 3: 'B-PERSON', 4: 'I-LOCATION', 5: 'I-MISCELLANEOUS', 6: 'I-ORGANIZATION', 7: 'I-PERSON', 8: 'O'}
Batch size: 16


In [17]:
@torch.no_grad()
def electra_ner(text: str, min_score: float = 0.70) -> List[Tuple[int, int, str, float]]:
    """
    Chạy Electra NER trên 1 text (dùng cho test/debug).
    Batch processing dùng batch_electra_ner() ở cell sau.
    """
    if not isinstance(text, str) or not text.strip():
        return []

    MAX_CHARS = 1500
    chunks = []
    if len(text) <= MAX_CHARS:
        chunks = [(text, 0)]
    else:
        sentences = re.split(r'(?<=[.!?])\s+', text)
        current_chunk, current_offset = "", 0
        for sent in sentences:
            if len(current_chunk) + len(sent) + 1 > MAX_CHARS and current_chunk:
                chunks.append((current_chunk, current_offset))
                current_offset += len(current_chunk) + 1
                current_chunk = sent
            else:
                current_chunk = (current_chunk + " " + sent) if current_chunk else sent
        if current_chunk:
            chunks.append((current_chunk, current_offset))

    results = []
    for chunk_text, offset in chunks:
        try:
            ner_results = ner_pipeline(chunk_text)
        except Exception:
            continue

        for ent in ner_results:
            mapped = ELECTRA_LABEL_MAP.get(ent["entity_group"], ent["entity_group"])
            if ent["score"] < min_score or mapped == "MISCELLANEOUS":
                continue
            results.append((ent["start"] + offset, ent["end"] + offset, mapped, ent["score"]))

    return results


# Test — chạy trên GPU
test_text = (
    "Ngày 16/1, ông Roland Staub - Chủ tịch Quỹ đầu tư Finance Suisse và "
    "ông Timur Mohamed - Chủ tịch công ty Palmer Johnson đến thăm Bình Định."
)
print(f"Input: {test_text}\n")
print(f"Electra NER results (on {torch.cuda.get_device_name(0)}):")
for start, end, etype, score in electra_ner(test_text):
    print(f"  [{start:3d}:{end:3d}] {etype:15s} ({score:.3f}) │ '{test_text[start:end]}'")

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Input: Ngày 16/1, ông Roland Staub - Chủ tịch Quỹ đầu tư Finance Suisse và ông Timur Mohamed - Chủ tịch công ty Palmer Johnson đến thăm Bình Định.

Electra NER results (on NVIDIA GeForce RTX 3070 Laptop GPU):
  [ 15: 27] PERSON          (0.999) │ 'Roland Staub'
  [ 39: 49] ORGANIZATION    (0.952) │ 'Quỹ đầu tư'
  [ 50: 64] ORGANIZATION    (0.965) │ 'Finance Suisse'
  [ 72: 85] PERSON          (0.999) │ 'Timur Mohamed'
  [105:119] ORGANIZATION    (0.908) │ 'Palmer Johnson'
  [129:138] LOCATION        (0.999) │ 'Bình Định'


## Layer 2: Regex Patterns

Nhận diện entity có cấu trúc rõ ràng: **MONEY**, **DATE**, **PERCENT**, **PRODUCT** (device families)

- `PRODUCT` dùng regex family thay vì liệt kê từng model → tự cover model mới ra mắt mà không cần update gazetteer

In [18]:
CURRENCY = r'(?:USD|VND|VNĐ|đồng|đô\s*la|CHF|EUR|JPY|CNY|nhân\s+dân\s+tệ)'
UNIT     = r'(?:triệu|tỷ|nghìn|ngàn|trăm)'
NUM      = r'\d[\d.,]*'

MONEY_PATTERNS = [
    re.compile(rf'\b{NUM}\s*{UNIT}\s*{CURRENCY}\b', re.IGNORECASE),
    re.compile(rf'\b{NUM}\s*{CURRENCY}\b', re.IGNORECASE),
    re.compile(rf'\$\s*{NUM}\b'),
    re.compile(rf'\b{NUM}\s*{UNIT}(?=\s|,|\.|\)|$)', re.IGNORECASE),
]

DATE_PATTERNS = [
    # Ngày/tháng với dấu gạch chéo
    re.compile(r'\b[Nn]gày\s+\d{1,2}/\d{1,2}(?:/\d{2,4})?\b'),
    re.compile(r'\b[Tt]háng\s+\d{1,2}(?:/\d{2,4})?\b'),
    re.compile(r'\b[Nn]ăm\s+\d{4}\b'),
    re.compile(r'\b\d{4}-\d{2}-\d{2}\b'),
    re.compile(r'(?<!\w)\d{1,2}/\d{1,2}/\d{2,4}(?!\w)'),
    # Quý — với và không có năm, cả chữ số Ả Rập lẫn La Mã
    re.compile(r'\b[Qq]uý\s+[1-4](?:\s*/\s*\d{4})?\b'),
    re.compile(r'\b[Qq]uý\s+[IViv]{1,4}(?:\s*/\s*\d{4})?\b'),
    re.compile(r'\bQ\s*[1-4]\s*/\s*\d{4}\b'),
    # Nửa năm / X tháng đầu/cuối năm (phổ biến trong báo cáo tài chính)
    re.compile(r'\bnửa\s+(?:đầu|cuối)\s+(?:năm\s+)?\d{4}\b', re.IGNORECASE),
    re.compile(r'\b\d{1,2}\s+tháng\s+(?:đầu|cuối|đầu\s+năm|cuối\s+năm)\s+\d{4}\b', re.IGNORECASE),
    re.compile(r'\b(?:6|9|10|11|12)\s+tháng\s+(?:đầu|cuối)\s+năm\s+\d{4}\b', re.IGNORECASE),
    # Phạm vi năm: 2024–2025, 2024/2025, 2023-2024
    re.compile(r'\b\d{4}\s*[-–]\s*\d{4}\b'),
    re.compile(r'\b\d{4}\s*/\s*\d{4}\b'),
    # Giai đoạn / kế hoạch X năm
    re.compile(r'\bgiai\s+đoạn\s+\d{4}\s*[-–]\s*\d{4}\b', re.IGNORECASE),
    re.compile(r'\b(?:kế\s+hoạch|chiến\s+lược|lộ\s+trình)\s+\d+\s+năm\b', re.IGNORECASE),
    # Thứ + ngày (lịch họp, sự kiện): "thứ Hai ngày 12/5"
    re.compile(
        r'\b[Tt]hứ\s+(?:[Hh]ai|[Bb]a|[Tt]ư|[Nn]ăm|[Ss]áu|[Bb]ảy|[Cc]hủ\s+nhật)'
        r'(?:\s+ngày\s+\d{1,2}/\d{1,2}(?:/\d{2,4})?)?\b'
    ),
    # Mùa trong năm
    re.compile(
        r'\b(?:đầu|cuối|giữa|mùa\s+hè|mùa\s+thu|mùa\s+đông|mùa\s+xuân)\s+năm\s+\d{4}\b',
        re.IGNORECASE
    ),
    # Năm học / tài khoá / học kỳ
    re.compile(r'\b(?:năm\s+học|niên\s+học|tài\s+khoá|tài\s+khóa|học\s+kỳ)\s+\d{4}(?:\s*[-–]\s*\d{4})?\b', re.IGNORECASE),
]

PERCENT_PATTERNS = [
    # Số + % đơn thuần (có/không /năm, /tháng, /quý)
    re.compile(r'(?<!\w)\d[\d.,]*\s*%(?:/(?:năm|tháng|quý))?(?!\w)'),
    # Âm X% (lãi suất âm, tăng trưởng âm)
    re.compile(r'\bâm\s+\d[\d.,]*\s*%(?:/năm)?(?!\w)', re.IGNORECASE),
    # ±X%
    re.compile(r'(?<!\w)[+\-]\s*\d[\d.,]*\s*%(?:/năm)?(?!\w)'),
    # X%–Y% (dải)
    re.compile(r'(?<!\w)\d[\d.,]*\s*%?\s*[-\u2013]\s*\d[\d.,]*\s*%(?!\w)'),
    # X phần trăm (chữ)
    re.compile(r'(?<!\w)\d[\d.,]*\s+phần\s+trăm(?!\w)', re.IGNORECASE),
    # Động từ + X% (báo cáo tài chính)
    re.compile(
        r'(?:tăng|giảm|đạt|chiếm|lên|xuống|khoảng|ước|thêm|còn|về|vượt|hơn)'
        r'\s+\d[\d.,]*\s*%(?:/(?:năm|tháng|quý))?',
        re.IGNORECASE
    ),
    # X điểm % / X điểm phần trăm (basis point kiểu VN)
    re.compile(r'(?<!\w)\d[\d.,]*\s+điểm\s*%(?!\w)', re.IGNORECASE),
    re.compile(r'(?<!\w)\d[\d.,]*\s+điểm\s+phần\s+trăm(?!\w)', re.IGNORECASE),
    # Xấp xỉ / gần / hơn / dưới / trên X%
    re.compile(
        r'(?:xấp\s+xỉ|gần|hơn|dưới|trên|vượt|đạt\s+mức|chỉ|mới)\s+'
        r'\d[\d.,]*\s*%',
        re.IGNORECASE
    ),
    # CAR/NIM/ROA/ROE X% (chỉ số ngân hàng viết liền)
    re.compile(r'\b(?:CAR|NIM|ROA|ROE|NPL|LDR|CIR)\s+\d[\d.,]*\s*%(?!\w)', re.IGNORECASE),
]

EVENT_PATTERNS = [
    # ── Khí hậu / môi trường ──────────────────────────────────
    re.compile(r'\bCOP\s*\d{2}\b'),
    # ── Thể thao — với/không năm ─────────────────────────────
    re.compile(r'\bSEA\s+Games(?:\s+\d{2,4})?\b', re.IGNORECASE),
    re.compile(r'\bOlympic(?:\s+\w+)?(?:\s+\d{4})?\b', re.IGNORECASE),
    re.compile(r'\b(?:FIFA\s+)?World\s+Cup(?:\s+\d{4})?\b', re.IGNORECASE),
    re.compile(r'\b(?:AFF\s+Cup|ASEAN\s+Cup|Asian\s+Cup|Asian\s+Games|ASIAD)(?:\s+\d{4})?\b', re.IGNORECASE),
    re.compile(r'\bU\s*\d{2}\s+(?:châu\s+Á|Asian|World\s+Cup|AFF)(?:\s+\d{4})?\b', re.IGNORECASE),
    re.compile(r'\b(?:Champions\s+League|Europa\s+League)(?:\s+\d{4}(?:-\d{2,4})?)?\b', re.IGNORECASE),
    re.compile(r'\bF1\s+(?:Grand\s+Prix|GP)(?:\s+[\w]+)?(?:\s+\d{4})?\b', re.IGNORECASE),
    # ── Global tech events — không bắt buộc năm ──────────────
    re.compile(
        r'\b(?:WWDC|MWC|CES|Google\s+I/O|Samsung\s+Unpacked|Samsung\s+Galaxy\s+Unpacked|'
        r'Meta\s+Connect|AWS\s+re:?Invent|Microsoft\s+(?:Ignite|Build)|'
        r'Qualcomm\s+Summit|Snapdragon\s+Summit|Apple\s+(?:Special\s+)?Event|'
        r'Apple\s+Worldwide\s+Developers\s+Conference)(?:\s+\d{4})?\b',
        re.IGNORECASE
    ),
    # ── Vietnamese digital/tech/startup events ────────────────
    re.compile(
        r'\b(?:Vietnam\s+Mobile\s+Day|Vietnam\s+Internet\s+Day|'
        r'Vietnam\s+Innovation\s+Summit|Vietnam\s+AI\s+(?:Summit|Day)|'
        r'Vietnam\s+Fintech\s+(?:Fest|Festival|Forum)|Vietnam\s+(?:Tech|Digital)\s+(?:Summit|Forum|Day)|'
        r'Vietnam\s+ePayment|Vietnam\s+Blockchain\s+Week|'
        r'Make\s+in\s+Viet\s+Nam|Make\s+in\s+Vietnam|'
        r'Techfest\s+Vietnam|Tech\s+Fest\s+Vietnam|ICT\s+Summit\s+Vietnam|'
        r'Ngày\s+Chuyển\s+đổi\s+số\s+(?:quốc\s+gia)?)(?:\s+\d{4})?\b',
        re.IGNORECASE
    ),
    # ── Fintech / ngân hàng / tài chính ──────────────────────
    re.compile(
        r'\b(?:Vietnam\s+(?:Banking|Finance|Financial)\s+(?:Forum|Summit|Outlook|Conference)|'
        r'Open\s+Banking\s+Forum|Vietnam\s+Fintech\s+Forum|Diễn\s+đàn\s+Fintech\s+Việt\s+Nam|'
        r'VNBA\s+(?:Annual\s+)?(?:Conference|Forum)|'
        r'Banking\s+Innovation\s+Summit)(?:\s+\d{4})?\b',
        re.IGNORECASE
    ),
    # ── Sự kiện chính phủ / quốc hội ─────────────────────────
    re.compile(r'\bHội\s+nghị\s+(?:Trung\s+ương|TW)\s+(?:lần\s+thứ\s+)?\d+\b', re.IGNORECASE),
    re.compile(r'\bĐại\s+hội\s+(?:[Đđ]ảng\s+)?(?:(?:toàn\s+quốc\s+)?lần\s+thứ\s+)?(?:[XIV]+|\d+)\b', re.IGNORECASE),
    re.compile(r'\bKỳ\s+họp\s+(?:thứ\s+)?\d+(?:\s*,?\s*Quốc\s+hội)?\b', re.IGNORECASE),
    re.compile(r'\bHội\s+nghị\s+(?:Chính\s+phủ|toàn\s+quốc)(?:\s+\d{4})?\b', re.IGNORECASE),
    # ── Giải thưởng + năm ─────────────────────────────────────
    re.compile(r'\bGiải\s+thưởng\s+(?:[\w.,\-]+\s+){1,6}\d{4}\b', re.IGNORECASE),
    re.compile(r'\b(?:Sao\s+Khuê|VinFuture|Vietnam\s+ICT\s+Awards?|Nhân\s+tài\s+Đất\s+Việt)(?:\s+\d{4})?\b', re.IGNORECASE),
    # ── Summit / Forum / Expo / Conference có năm ─────────────
    re.compile(r'\b(?:[\w]+\s+){1,5}(?:Summit|Forum|Expo|Conference|Congress|Festival)\s+\d{4}\b', re.IGNORECASE),
    # ── Ngày hội (lễ hội mua sắm, ngày hội công nghệ...) ─────
    re.compile(r'\bNgày\s+hội\s+(?:[\w]+\s+){1,4}(?:\d{4}\b|(?=\n|$))', re.IGNORECASE),
    # ── E-commerce sales events ───────────────────────────────
    re.compile(r'\b(?:11\.11|12\.12|10\.10|9\.9|8\.8)\b'),
    re.compile(r'\b(?:Black\s+Friday|Cyber\s+Monday|Double\s+Day)(?:\s+\d{4})?\b', re.IGNORECASE),
    # ── Regional / quốc tế ───────────────────────────────────
    re.compile(r'\b(?:APEC|ASEAN\s+Summit|G7|G20|WEF|Davos|RCEP|AIIB)(?:\s+\d{4})?\b', re.IGNORECASE),
    re.compile(r'\bDiễn\s+đàn\s+Kinh\s+tế\s+(?:Việt\s+Nam|Thế\s+giới)(?:\s+\d{4})?\b', re.IGNORECASE),
    # ── Triển lãm / trade shows ──────────────────────────────
    re.compile(r'\b(?:COMPUTEX|Gitex|IFA|NAB\s+Show|Interop)(?:\s+\d{4})?\b', re.IGNORECASE),
    # ── Startup / ecosystem events ───────────────────────────
    re.compile(r'\b(?:Shark\s+Tank\s+Việt\s+Nam|Shark\s+Tank\s+Vietnam|Vietnam\s+Expo)(?:\s+\d{4})?\b', re.IGNORECASE),
]

# === PRODUCT regex families ===
# Dùng pattern thay vì liệt kê từng model → tự cover model mới ra mắt
PRODUCT_PATTERNS = [
    # iPhone 15, iPhone 17 Pro, iPhone 17 Pro Max, iPhone Air, iPhone SE 4
    re.compile(r'\biPhone\s+(?:\d{1,2}(?:\s+(?:Pro(?:\s+Max)?|Plus|Air|Mini))?|Air|SE(?:\s+\d)?)\b'),
    # Samsung Galaxy S25, Galaxy A55, Galaxy Z Fold 7, Galaxy Z Flip 7, Galaxy Tab S10
    re.compile(r'\bGalaxy\s+(?:[A-Z]\d{1,2}\+?|Z\s+(?:Fold|Flip)\s*\d|Tab\s+S\d+)\b'),
    # Google Pixel 9, Pixel 9 Pro, Pixel 9 Pro XL, Pixel 9a, Pixel 9 Fold
    re.compile(r'\bPixel\s+\d(?:\s+(?:Pro(?:\s+XL)?|Fold|a))?\b'),
    # macOS Sonoma, macOS Tahoe, macOS 15.1
    re.compile(r'\bmacOS\s+(?:[A-Z][a-z]+|\d{1,2}(?:\.\d+)?)\b'),
    # iOS 17, iOS 26.1 / iPadOS 18 / watchOS 11 / tvOS 18
    re.compile(r'\bi(?:OS|PadOS|watchOS|tvOS)\s+\d{1,2}(?:\.\d+)?\b'),
    # Android 14, Android 15.1
    re.compile(r'\bAndroid\s+\d{1,2}(?:\.\d+)?\b'),
    # Snapdragon 8 Gen 3, Snapdragon X Elite, Snapdragon X Plus
    re.compile(r'\bSnapdragon\s+(?:\d+\s+Gen\s+\d|X\s+(?:Elite|Plus))\b'),
    # Dimensity 9300, Dimensity 9400 / Helio G99
    re.compile(r'\b(?:Dimensity|Helio)\s+[A-Z]?\d{2,4}\b'),
    # Razr 50, Razr 50 Ultra, Razr 2025
    re.compile(r'\bRazr\s+(?:\d{2,4}(?:\s+(?:Ultra|Plus|s))?)\b'),
    # AirPods Pro 2, AirPods 4
    re.compile(r'\bAirPods\s+(?:Pro\s*\d?|\d)\b'),
    # Apple Watch Series 11, Apple Watch Ultra 3, Apple Watch SE 3
    re.compile(r'\bApple\s+Watch\s+(?:Series\s+\d+|Ultra\s*\d?|SE\s*\d?)\b'),
]


def regex_ner(text: str) -> List[Tuple[int, int, str]]:
    """
    Tìm MONEY, DATE, PERCENT, PRODUCT bằng regex.
    - PRODUCT dùng family patterns → cover model mới không cần update gazetteer
    - Dedup: cùng (start, end) chỉ giữ 1 match (ưu tiên match đầu tiên)
    """
    if not isinstance(text, str):
        return []

    spans = []
    seen: set = set()

    def _add(start: int, end: int, etype: str):
        key = (start, end)
        if key not in seen:
            seen.add(key)
            spans.append((start, end, etype))

    for patterns, etype in [
        (MONEY_PATTERNS,   "MONEY"),
        (DATE_PATTERNS,    "DATE"),
        (PERCENT_PATTERNS, "PERCENT"),
        (EVENT_PATTERNS,   "EVENT"),
        (PRODUCT_PATTERNS, "PRODUCT"),
    ]:
        for pat in patterns:
            for m in pat.finditer(text):
                _add(m.start(), m.end(), etype)

    return spans


# Test
test_text = "Ngày 16/1, GDP tăng 6,5% trong năm 2024. Giá 1000 USD, tương đương 12 triệu đồng."
print(f"Input: {test_text}\n")
for s, e, t in regex_ner(test_text):
    print(f"  [{s:3d}:{e:3d}] {t:10s} │ '{test_text[s:e]}'")

print("\n--- PRODUCT regex families ---")
for tc in [
    "Apple ra mắt iPhone 17 Pro Max và iPhone Air tại WWDC.",
    "Samsung tung Galaxy S25 Ultra và Galaxy Z Fold 7.",
    "Snapdragon 8 Gen 4 mạnh hơn Apple M3.",
    "iOS 26.1 fix bug, macOS Tahoe ra mắt tháng 9.",
    "AirPods Pro 3 và Apple Watch Series 11.",
]:
    print(f"\n  '{tc}'")
    for s, e, t in regex_ner(tc):
        if t == "PRODUCT":
            print(f"    [{s:3d}:{e:3d}] {t} │ '{tc[s:e]}'")

Input: Ngày 16/1, GDP tăng 6,5% trong năm 2024. Giá 1000 USD, tương đương 12 triệu đồng.

  [ 67: 80] MONEY      │ '12 triệu đồng'
  [ 45: 53] MONEY      │ '1000 USD'
  [ 67: 75] MONEY      │ '12 triệu'
  [  0:  9] DATE       │ 'Ngày 16/1'
  [ 31: 39] DATE       │ 'năm 2024'
  [ 20: 24] PERCENT    │ '6,5%'
  [ 15: 24] PERCENT    │ 'tăng 6,5%'

--- PRODUCT regex families ---

  'Apple ra mắt iPhone 17 Pro Max và iPhone Air tại WWDC.'
    [ 13: 30] PRODUCT │ 'iPhone 17 Pro Max'
    [ 34: 44] PRODUCT │ 'iPhone Air'

  'Samsung tung Galaxy S25 Ultra và Galaxy Z Fold 7.'
    [ 13: 23] PRODUCT │ 'Galaxy S25'
    [ 33: 48] PRODUCT │ 'Galaxy Z Fold 7'

  'Snapdragon 8 Gen 4 mạnh hơn Apple M3.'
    [  0: 18] PRODUCT │ 'Snapdragon 8 Gen 4'

  'iOS 26.1 fix bug, macOS Tahoe ra mắt tháng 9.'
    [ 18: 29] PRODUCT │ 'macOS Tahoe'
    [  0:  8] PRODUCT │ 'iOS 26.1'

  'AirPods Pro 3 và Apple Watch Series 11.'
    [  0: 13] PRODUCT │ 'AirPods Pro 3'
    [ 17: 38] PRODUCT │ 'Apple Watch Series 11'


## Layer 3: Gazetteer Dictionary

Vai trò kép:
- **Nhận diện entity model không cover:** PRODUCT, EVENT, INDUSTRY
- **Bổ sung entity model bỏ sót:** tên viết tắt (ACB, BIDV), tên nước ngoài (Elon Musk)...

Cải tiến:
- **PRODUCT tách HARDWARE + SOFTWARE** → granular hơn (chip, device vs app, AI model)
- **Aho-Corasick automaton** → O(n) matching, nhanh ~20x so với str.find loop
- **Contextual INDUSTRY filter** → chỉ tag "game", "công nghệ"... khi có trigger word gần đó
- **Pre-sort** dài→ngắn đảm bảo longest match first

In [19]:
# ════════════════════════════════════════════════════════════
# GAZETTEER — entity mà model KHÔNG cover hoặc hay bỏ sót
# Tổ chức theo domain: HARDWARE, SOFTWARE, PLATFORM, STOCK,
#   EVENT, INDUSTRY, ORGANIZATION, LOCATION, PERSON
# ════════════════════════════════════════════════════════════

# ── PRODUCT: chia thành HARDWARE và SOFTWARE (label đều = PRODUCT) ──
GAZ_HARDWARE = [
    # Apple devices (chỉ giữ tên không có số — các model có số đã cover bởi regex)
    "AirPods", "HomePod", "HomeHub", "Apple Watch",
    # Motorola
    "Razr",
    # Network tech
    "5G Advanced", "5G SA", "5G NSA", "LTE Advanced", "MIMO",
    # Chips (generic)
    "Snapdragon", "MediaTek", "Apple M3", "Apple M4",
]

GAZ_SOFTWARE = [
    # AI assistants & models
    "ChatGPT", "Gemini", "Siri", "Alexa", "Copilot",
    "Claude", "DeepSeek", "Llama", "Mistral",
    "Apple Intelligence", "Apple Foundation Models",
    "Google Gemini", "Microsoft Copilot",
    # Platforms & stores
    "App Store", "Google Play",
    "Open Banking", "OneBank", "OneBank 365+",
    "CheckVN", "VDAPES", "vTravel",
    # Vietnamese apps
    "ZaloPay", "MoMo", "VNPay", "VietQR",
    "Shopee", "Lazada", "Tiki", "TikTok Shop",
]

GAZ_EVENT = [
    # ── Global tech events (có và không có năm) ──────────────
    "WWDC", "WWDC 2024", "WWDC 2025", "WWDC 2026",
    "Apple Worldwide Developers Conference",
    "Google I/O", "Google I/O 2024", "Google I/O 2025",
    "MWC", "MWC 2024", "MWC 2025", "MWC 2026",
    "CES", "CES 2024", "CES 2025", "CES 2026",
    "Samsung Unpacked", "Samsung Galaxy Unpacked",
    "Samsung Unpacked 2024", "Samsung Unpacked 2025",
    "Meta Connect", "Meta Connect 2024", "Meta Connect 2025",
    "AWS re:Invent", "AWS re:Invent 2024", "AWS re:Invent 2025",
    "Microsoft Ignite", "Microsoft Ignite 2024", "Microsoft Ignite 2025",
    "Microsoft Build", "Microsoft Build 2024", "Microsoft Build 2025",
    "Apple Event", "Apple Special Event",
    "Qualcomm Summit", "Snapdragon Summit",
    "Hội nghị các nhà phát triển toàn cầu",
    "Google Cloud Next", "Google Cloud Next 2024", "Google Cloud Next 2025",
    "Meta AI Summit", "OpenAI DevDay", "Anthropic Summit",
    "Nvidia GTC", "Nvidia GTC 2024", "Nvidia GTC 2025",

    # ── Vietnam digital / tech / startup events ──────────────
    "Vietnam Mobile Day", "Vietnam Mobile Day 2024", "Vietnam Mobile Day 2025",
    "Vietnam Internet Day", "Vietnam Internet Day 2024",
    "Vietnam Blockchain Week",
    "Vietnam AI Summit", "Vietnam AI Day",
    "Vietnam Tech Summit", "Vietnam Tech Summit 2024", "Vietnam Tech Summit 2025",
    "Vietnam Innovation Summit",
    "Vietnam Fintech Fest", "Vietnam Fintech Festival",
    "Vietnam Fintech Forum", "Diễn đàn Fintech Việt Nam",
    "Vietnam ePayment", "Vietnam ePayment 2024", "Vietnam ePayment 2025",
    "Vietnam Digital Forum", "Vietnam Digital Economy Forum",
    "Vietnam Startup Festival",
    "Smart Vietnam Forum", "Digital Economy Summit",
    "ICT Summit", "ICT Summit Vietnam", "ICT Summit Vietnam 2024",
    "Tech Fest", "Tech Fest Vietnam", "Techfest Vietnam",
    "Techfest Vietnam 2024", "Techfest Vietnam 2025",
    "Make in Viet Nam", "Make in Vietnam",
    "Hanoi Innovation Summit",
    "Ngày Chuyển đổi số quốc gia",
    "Diễn đàn chuyển đổi số",
    "Chuyển đổi số Quốc gia",
    "Gaming On TikTok Hanoi Summit 2025",
    "Shark Tank Việt Nam", "Shark Tank Vietnam",
    "Vietnam Expo", "Vietnam International Expo",

    # ── Banking / tài chính / bảo hiểm ───────────────────────
    "Vietnam Banking Outlook", "Vietnam Banking Outlook 2025",
    "Vietnam Banking Forum", "Vietnam Banking Forum 2024", "Vietnam Banking Forum 2025",
    "Hội nghị ngân hàng", "Hội nghị Ngân hàng thường niên",
    "VNBA Annual Conference", "VNBA Forum",
    "Open Banking Forum", "Open Banking Forum Việt Nam",
    "Vietnam Financial Forum", "Vietnam Finance Summit",
    "Banking Innovation Summit", "Banking Technology Summit",
    "Vietnam Insurance Forum", "Diễn đàn Bảo hiểm Việt Nam",
    "Vietnam Capital Market Forum",
    "Vietnam Bond Market Forum",
    "Vietnam Investment Forum",

    # ── Chứng khoán / thị trường vốn ─────────────────────────
    "Vietnam Investor Day",
    "Vietnam M&A Forum", "Vietnam M&A Forum 2024", "Vietnam M&A Forum 2025",

    # ── Chính phủ / Quốc hội / Đảng ─────────────────────────
    "Đại hội Đảng XIII", "Đại hội Đảng XIV",
    "Đại hội toàn quốc lần thứ XIII", "Đại hội toàn quốc lần thứ XIV",
    "Hội nghị Trung ương 10", "Hội nghị Trung ương 11",
    "Hội nghị Trung ương 12", "Hội nghị Trung ương 13",
    "Hội nghị Trung ương 14", "Hội nghị Trung ương 15",
    "Kỳ họp thứ 8 Quốc hội", "Kỳ họp thứ 9 Quốc hội",
    "Kỳ họp thứ 10 Quốc hội", "Kỳ họp bất thường Quốc hội",
    "Hội nghị Chính phủ",
    "Hội nghị toàn quốc",
    "Hội nghị sơ kết", "Hội nghị tổng kết",

    # ── Giải thưởng & công nhận ──────────────────────────────
    "Giải thưởng Sao Khuê", "Sao Khuê",
    "Sao Khuê 2024", "Sao Khuê 2025",
    "Giải thưởng VinFuture", "VinFuture", "VinFuture 2024", "VinFuture 2025",
    "Giải thưởng Nhân tài Đất Việt",
    "Nhân tài Đất Việt 2024", "Nhân tài Đất Việt 2025",
    "Vietnam ICT Awards", "Vietnam ICT Awards 2024", "Vietnam ICT Awards 2025",
    "Vietnam Excellence Award", "Vietnam Excellence Awards",
    "Vietnam Most Admired Companies",
    "Forbes 30 Under 30", "Forbes Vietnam",
    "Top 50 Forbes Vietnam",
    "HR Asia Awards", "HR Asia Awards 2024", "HR Asia Awards 2025",
    "Great Place to Work", "Best Companies to Work For",
    "Giải thưởng Chuyển đổi số Việt Nam",
    "Vietnam Digital Award", "Vietnam Digital Awards",
    "Vietnam Digital Awards 2024",
    "Giải thưởng Thương hiệu Vàng", "Vietnam Golden Brand",
    "Top 10 Doanh nghiệp CNTT Việt Nam",
    "ASEAN Business Awards",

    # ── Kinh tế / doanh nghiệp ───────────────────────────────
    "Vietnam CEO Forum",
    "Vietnam Business Forum", "Vietnam Business Forum 2024",
    "Diễn đàn Kinh tế Việt Nam",
    "Diễn đàn Kinh tế Việt Nam 2024", "Diễn đàn Kinh tế Việt Nam 2025",
    "Diễn đàn Doanh nghiệp Việt Nam",
    "Vietnam Economic Forum",
    "World Economic Forum", "WEF",
    "Davos", "Davos 2024", "Davos 2025",
    "G7", "G20", "G7 Summit", "G20 Summit",
    "G7 Summit 2025", "G20 Summit 2025",

    # ── Khu vực / quốc tế ────────────────────────────────────
    "APEC", "APEC 2024", "APEC 2025",
    "ASEAN Summit", "ASEAN Summit 2024", "ASEAN Summit 2025",
    "ASEAN Leaders Summit",
    "ASEAN Digital Ministers Meeting",
    "Belt and Road Forum", "Belt and Road Forum 2023",
    "RCEP", "AIIB",
    "Hội nghị Thượng đỉnh ASEAN",

    # ── E-commerce & bán lẻ ──────────────────────────────────
    "11.11", "12.12", "10.10", "9.9", "8.8",
    "Ngày hội mua sắm", "Ngày Độc thân",
    "Lễ hội mua sắm 11.11",
    "Black Friday", "Cyber Monday",
    "Shopee Super Sale", "Lazada Sale",

    # ── Văn hoá / lễ tết ─────────────────────────────────────
    "Tết Nguyên Đán", "Tết Nguyên Đán 2025", "Tết Nguyên Đán 2026",
    "Tết Ất Tỵ", "Tết Bính Ngọ", "Tết Đinh Mùi",
    "vía Thần Tài", "Lễ Giỗ Tổ Hùng Vương",
    "Quốc khánh", "Ngày Quốc tế Phụ nữ",
    "Ngày Nhà giáo Việt Nam", "Ngày Thầy thuốc Việt Nam",

    # ── Thể thao ──────────────────────────────────────────────
    "SEA Games", "SEA Games 32", "SEA Games 33",
    "ASIAD", "ASIAD 19", "ASIAD 20",
    "Olympic", "Olympic 2024", "Olympic Paris 2024", "Olympic 2028",
    "World Cup", "World Cup 2026", "FIFA World Cup 2026",
    "AFF Cup", "AFF Cup 2024", "AFF Cup 2026",
    "Asian Cup", "Asian Cup 2023", "Asian Cup 2027",
    "Champions League", "Europa League",
    "Asian Games",
    "Vietnam Open", "VPBank Hanoi Marathon",

    # ── Môi trường / khí hậu ─────────────────────────────────
    "COP26", "COP27", "COP28", "COP29", "COP30",
    "Net Zero 2050", "Hội nghị biến đổi khí hậu",

    # ── Triển lãm ─────────────────────────────────────────────
    "COMPUTEX", "COMPUTEX 2024", "COMPUTEX 2025",
    "Mobile World Congress",
    "Vietnam Expo 2024", "Vietnam Expo 2025",
    "Triển lãm Quốc tế Việt Nam",
]

# INDUSTRY: chỉ match khi có context trigger (xem INDUSTRY_CONTEXT_TRIGGERS)
# → Giảm false positive cho term ngắn như "game", "công nghệ"
GAZ_INDUSTRY = [
    # Multi-word terms (ít FP, không cần context)
    "nông nghiệp công nghệ cao", "thương mại điện tử xuyên biên giới",
    "công nghiệp bán dẫn", "công nghiệp game", "ngành game",
    "phòng cháy chữa cháy", "năng lượng tái tạo", "năng lượng sạch",
    "trí tuệ nhân tạo", "chuyển đổi số", "kinh tế số",
    # Single/short terms (cần context để tránh FP)
    "bất động sản", "ngân hàng", "tài chính", "chứng khoán",
    "công nghệ", "viễn thông", "thương mại điện tử",
    "du lịch", "hàng không", "logistics", "vận tải",
    "nông nghiệp", "công nghiệp", "y tế", "giáo dục",
    "xây dựng", "năng lượng", "dầu khí", "bảo hiểm",
    "game", "điện ảnh", "thời trang", "ẩm thực",
]

# Context triggers: INDUSTRY chỉ được match khi có trigger trong window 50 chars trước
INDUSTRY_CONTEXT_TRIGGERS = frozenset({
    "ngành", "lĩnh vực", "thị trường", "thị phần", "sector",
    "công nghiệp", "ngành hàng", "phân khúc", "hệ sinh thái",
})

# INDUSTRY terms ngắn cần context (các term dài ≥3 từ tự match không cần)
INDUSTRY_NEEDS_CONTEXT = frozenset({
    "công nghệ", "game", "y tế", "giáo dục", "du lịch",
    "ngân hàng", "tài chính", "logistics", "vận tải",
    "nông nghiệp", "công nghiệp", "xây dựng", "năng lượng",
    "dầu khí", "bảo hiểm", "điện ảnh", "thời trang", "ẩm thực",
    "chứng khoán", "hàng không", "viễn thông",
})

GAZ_ORGANIZATION_PATCH = [
    # Vietnamese banks
    "BIDV", "ACB", "SHB", "OCB", "MSB", "VIB", "NCB", "KLB",
    "HDBank", "LPBank", "TPBank", "VPBank", "ABBank", "BVBank",
    "SeABank", "PGBank", "VietBank", "Saigonbank", "KienlongBank",
    "Bac A Bank", "Viet A Bank", "Nam A Bank",
    # Vietnamese exchanges & regulators
    "HoSE", "HNX", "UPCOM", "NAPAS", "NHNN", "UBND",
    # Vietnamese tech companies
    "Viettel", "VNPT", "FPT", "VNG", "MoMo", "ZaloPay",
    "VinAI", "VinBigData", "Tiki", "Sendo",
    "VTC", "VTC Pay", "VNPT VinaPhone",
    # International
    "SECO", "WHO", "UNESCO", "ASEAN", "EU", "NATO",
    "IMF", "WTO", "APEC", "3GPP", "GSMA",
    # Global tech (model hay miss)
    "Nvidia", "TSMC", "Qualcomm", "ARM", "MediaTek",
    "ByteDance", "Tencent", "Alibaba", "Baidu", "Huawei",
    "OpenAI", "Anthropic", "Mistral AI", "DeepSeek",
    # Research & media
    "Finance Suisse", "Palmer Johnson", "Bloomberg Intelligence",
    "Sensor Tower", "Statista", "LightReading", "The Verge", "HR Asia",
    "IDC", "Gartner", "Forrester",
    # Vietnamese gov departments
    "Cục Thương mại Điện tử và Kinh tế số",
    "Cục Quản lý và Phát triển thị trường trong nước",
    "Cục Công nghiệp CNTT",
    "Đại học Bách Khoa Hà Nội",
    "Tổng cục Kinh tế Liên bang Thụy Sĩ",
    "Phát Đạt Corporation",
]

GAZ_LOCATION_PATCH = [
    "Đông Nam Á", "Châu Á", "Tây Bắc", "Đông Bắc", "Trung Đông",
    "Thung lũng Silicon", "Silicon Valley",
    "Biển Đông", "Hoàng Sa", "Trường Sa",
    "Khu công nghệ cao Hòa Lạc", "KCN VSIP",
    "chợ đầu mối Bình Điền",
    "TP.HCM", "TP HCM",
]

GAZ_PERSON_PATCH = [
    "Mark Zuckerberg", "Zuckerberg", "Steve Jobs", "Tim Cook",
    "Elon Musk", "Jeff Bezos", "Sundar Pichai",
    "Andy Jassy", "Børje Ekholm", "Dan Ives",
    "Sam Altman", "Satya Nadella", "Jensen Huang",
    "Joe Rogan", "Rogan",
    "Tomasz Noetzel", "Tse King Kiu",
    "Park Hang Seo", "Park Hang-seo",
]

# ════════════════════════════════════════════════════════════
# Build sorted entries (PRE-SORT 1 lần, dài nhất trước)
# Mỗi entry: (norm_name, orig_name, etype, case_sensitive)
# ════════════════════════════════════════════════════════════

_RAW_GAZETTEERS = {
    "PRODUCT":      (GAZ_HARDWARE + GAZ_SOFTWARE, True),
    "EVENT":        (GAZ_EVENT, True),
    "INDUSTRY":     (GAZ_INDUSTRY, False),
    "ORGANIZATION": (GAZ_ORGANIZATION_PATCH, True),
    "LOCATION":     (GAZ_LOCATION_PATCH, True),
    "PERSON":       (GAZ_PERSON_PATCH, True),
}


def _build_sorted_entries(raw: dict) -> list:
    entries = []
    for etype, (gaz_list, case_sensitive) in raw.items():
        for name in gaz_list:
            norm = normalize_text(name)
            if norm:
                entries.append((norm, name, etype, case_sensitive))
    entries.sort(key=lambda x: -len(x[0]))
    return entries


SORTED_GAZETTEER = _build_sorted_entries(_RAW_GAZETTEERS)

# ════════════════════════════════════════════════════════════
# Build Aho-Corasick automaton (O(n) matching vs O(n×m) str.find)
# ════════════════════════════════════════════════════════════

def _build_automaton(entries: list):
    """
    Build Aho-Corasick từ gazetteer entries.
    Key = lowercase (hỗ trợ cả case-sensitive & insensitive).
    Value = list of (norm_name, orig_name, etype, case_sensitive).
    """
    if not HAS_AHOCORASICK:
        return None

    key_map: Dict[str, list] = {}
    for norm_name, orig_name, etype, case_sensitive in entries:
        key = norm_name.lower()
        if key not in key_map:
            key_map[key] = []
        key_map[key].append((norm_name, orig_name, etype, case_sensitive))

    A = ahocorasick.Automaton()
    for key, payloads in key_map.items():
        A.add_word(key, payloads)
    A.make_automaton()
    return A


GAZ_AUTOMATON = _build_automaton(SORTED_GAZETTEER)

# Stats
total = len(SORTED_GAZETTEER)
by_type = Counter(e[2] for e in SORTED_GAZETTEER)
print(f"Gazetteer: {total} entries {'(Aho-Corasick ✓)' if GAZ_AUTOMATON else '(str.find fallback)'}")
for etype, cnt in by_type.most_common():
    cs = next(e[3] for e in SORTED_GAZETTEER if e[2] == etype)
    print(f"  {etype:15s}: {cnt:3d} entries  (case_sensitive={cs})")
print(f"\nTop-5 dài nhất:")
for norm, orig, etype, _ in SORTED_GAZETTEER[:5]:
    print(f"  [{len(norm):3d} chars] {etype:15s} │ '{orig}'")

Gazetteer: 452 entries (Aho-Corasick ✓)
  EVENT          : 258 entries  (case_sensitive=True)
  ORGANIZATION   :  83 entries  (case_sensitive=True)
  PRODUCT        :  43 entries  (case_sensitive=True)
  INDUSTRY       :  34 entries  (case_sensitive=False)
  PERSON         :  19 entries  (case_sensitive=True)
  LOCATION       :  15 entries  (case_sensitive=True)

Top-5 dài nhất:
  [ 47 chars] ORGANIZATION    │ 'Cục Quản lý và Phát triển thị trường trong nước'
  [ 37 chars] EVENT           │ 'Apple Worldwide Developers Conference'
  [ 36 chars] EVENT           │ 'Hội nghị các nhà phát triển toàn cầu'
  [ 36 chars] ORGANIZATION    │ 'Cục Thương mại Điện tử và Kinh tế số'
  [ 34 chars] EVENT           │ 'Gaming On TikTok Hanoi Summit 2025'


In [20]:
_VI_WORD_CHARS = set(
    "aăâáắấàằầảẳẩãẵẫạặậeêéếèềẻểẽễẹệiíìỉĩị"
    "oôơóốớòồờỏổởõỗỡọộợuưúứùừủửũữụựyýỳỷỹỵđ"
    "AĂÂÁẮẤÀẰẦẢẲẨÃẴẪẠẶẬEÊÉẾÈỀẺỂẼỄẸỆIÍÌỈĨỊ"
    "OÔƠÓỐỚÒỒỜỎỔỞÕỖỠỌỘỢUƯÚỨÙỪỦỬŨỮỤỰYÝỲỶỸỴĐ"
)


def _is_word_char(ch: str) -> bool:
    return ch.isalnum() or ch in _VI_WORD_CHARS


def _check_boundary(text: str, start: int, end: int) -> bool:
    """Word boundary hỗ trợ tiếng Việt (regex \\b không hiểu dấu)."""
    before_ok = (start == 0 or not _is_word_char(text[start - 1]))
    after_ok   = (end >= len(text) or not _is_word_char(text[end]))
    return before_ok and after_ok


def _has_industry_context(text: str, start: int, window: int = 50) -> bool:
    """Kiểm tra có trigger word trong window trước entity → giảm FP cho INDUSTRY."""
    prefix = text[max(0, start - window):start].lower()
    suffix = text[start:min(len(text), start + window)].lower()
    return any(t in prefix or t in suffix for t in INDUSTRY_CONTEXT_TRIGGERS)


def _gazetteer_ner_ahocorasick(text: str) -> List[Tuple[int, int, str]]:
    """Matching dùng Aho-Corasick O(n) — nhanh ~20x so với str.find loop."""
    text_lower = text.lower()
    raw: List[Tuple[int, int, str, int]] = []  # (start, end, etype, length)

    for end_idx, payloads in GAZ_AUTOMATON.iter(text_lower):
        for norm_name, orig_name, etype, case_sensitive in payloads:
            key_len = len(norm_name)
            start_idx = end_idx - key_len + 1

            # Case-sensitive verification
            if case_sensitive and text[start_idx:end_idx + 1] != norm_name:
                continue

            if not _check_boundary(text, start_idx, end_idx + 1):
                continue

            # Contextual filter cho short INDUSTRY terms
            if etype == "INDUSTRY" and norm_name.lower() in INDUSTRY_NEEDS_CONTEXT:
                if not _has_industry_context(text, start_idx):
                    continue

            raw.append((start_idx, end_idx + 1, etype, key_len))

    # Longest match first → resolve overlaps
    raw.sort(key=lambda x: (-x[3], x[0]))
    spans, occupied = [], set()
    for start, end, etype, _ in raw:
        chars = set(range(start, end))
        if not chars & occupied:
            spans.append((start, end, etype))
            occupied |= chars

    spans.sort(key=lambda x: x[0])
    return spans


def _gazetteer_ner_fallback(text: str) -> List[Tuple[int, int, str]]:
    """Fallback dùng str.find khi không có pyahocorasick."""
    text_lower = text.lower()
    spans, occupied = [], set()

    for norm_name, orig_name, etype, case_sensitive in SORTED_GAZETTEER:
        search_text = text if case_sensitive else text_lower
        search_name = norm_name if case_sensitive else norm_name.lower()

        pos = 0
        while True:
            idx = search_text.find(search_name, pos)
            if idx == -1:
                break
            end = idx + len(search_name)

            if _check_boundary(text, idx, end):
                if etype == "INDUSTRY" and norm_name.lower() in INDUSTRY_NEEDS_CONTEXT:
                    if not _has_industry_context(text, idx):
                        pos = idx + 1
                        continue
                chars = set(range(idx, end))
                if not chars & occupied:
                    spans.append((idx, end, etype))
                    occupied |= chars

            pos = idx + 1

    spans.sort(key=lambda x: x[0])
    return spans


def gazetteer_ner(text: str) -> List[Tuple[int, int, str]]:
    """
    Tìm entity bằng gazetteer.
    - Aho-Corasick nếu có pyahocorasick (O(n)), ngược lại str.find (O(n×m))
    - Longest match first, boundary check hỗ trợ tiếng Việt
    - INDUSTRY short terms: yêu cầu context trigger trong window 50 chars
    """
    if not isinstance(text, str) or not text.strip():
        return []
    if GAZ_AUTOMATON is not None:
        return _gazetteer_ner_ahocorasick(text)
    return _gazetteer_ner_fallback(text)


# ── Tests ──
test_text = "Apple ra mắt iPhone 17 tại HR Asia Awards 2025, ngành công nghệ rất sôi động."
print(f"Input: {test_text}\n")
for s, e, t in gazetteer_ner(test_text):
    print(f"  [{s:3d}:{e:3d}] {t:15s} │ '{test_text[s:e]}'")

print("\n--- Overlap: iPhone 17 vs iPhone (dài hơn thắng) ---")
for s, e, t in gazetteer_ner("Tôi mua iPhone 17 và AirPods Pro 3."):
    print(f"  [{s:3d}:{e:3d}] {t:15s} │ '{('Tôi mua iPhone 17 và AirPods Pro 3.')[s:e]}'")

print("\n--- INDUSTRY contextual filter ---")
for tc in [
    "ngành công nghệ Việt Nam tăng trưởng mạnh",    # có trigger → match
    "đây là trò game đơn giản",                      # không trigger → NO match
    "lĩnh vực du lịch phục hồi tốt",                # có trigger → match
    "anh ấy làm game cả ngày",                       # không trigger → NO match
]:
    hits = [(t, tc[s:e]) for s, e, t in gazetteer_ner(tc) if t == "INDUSTRY"]
    print(f"  {'✓' if hits else '✗'} '{tc}' → {hits if hits else 'no INDUSTRY match'}")

Input: Apple ra mắt iPhone 17 tại HR Asia Awards 2025, ngành công nghệ rất sôi động.

  [ 27: 46] EVENT           │ 'HR Asia Awards 2025'
  [ 54: 63] INDUSTRY        │ 'công nghệ'

--- Overlap: iPhone 17 vs iPhone (dài hơn thắng) ---
  [ 21: 28] PRODUCT         │ 'AirPods'

--- INDUSTRY contextual filter ---
  ✓ 'ngành công nghệ Việt Nam tăng trưởng mạnh' → [('INDUSTRY', 'công nghệ')]
  ✗ 'đây là trò game đơn giản' → no INDUSTRY match
  ✓ 'lĩnh vực du lịch phục hồi tốt' → [('INDUSTRY', 'du lịch')]
  ✗ 'anh ấy làm game cả ngày' → no INDUSTRY match


In [21]:
# ════════════════════════════════════════════════════════════
# Canonicalization: map alias → canonical name
# Dùng trong JSON export để chuẩn hoá tên entity khi aggregate
# KHÔNG thay đổi BIO labels (BIO vẫn dùng surface form gốc)
# ════════════════════════════════════════════════════════════

CANONICAL_MAP: Dict[str, str] = {
    # LOCATION
    "TP HCM":   "TP.HCM",
    "TPHCM":    "TP.HCM",
    "Sài Gòn":  "TP.HCM",
    "Hà nội":   "Hà Nội",
    "ha noi":   "Hà Nội",
    # ORGANIZATION
    "Apple Inc.":        "Apple",
    "Apple Inc":         "Apple",
    "Meta Platforms":    "Meta",
    "Facebook Inc":      "Meta",
    "Facebook":          "Meta",
    "Alphabet Inc":      "Google",
    "Google LLC":        "Google",
    "Microsoft Corp":    "Microsoft",
    "Microsoft Corp.":   "Microsoft",
    "Mistral AI":        "Mistral",
    # PERSON
    "Elon R. Musk":     "Elon Musk",
    "Park Hang-seo":    "Park Hang Seo",
    "Park Hang Seo":    "Park Hang Seo",
    # PRODUCT aliases
    "Chat GPT":  "ChatGPT",
    "chat gpt":  "ChatGPT",
}


def canonicalize(surface: str) -> str:
    """Map surface form → canonical name (fallback: surface form gốc)."""
    return CANONICAL_MAP.get(surface, CANONICAL_MAP.get(surface.strip(), surface))


# Test
test_aliases = ["TP HCM", "Facebook", "Apple Inc.", "Chat GPT", "Hà nội", "Park Hang-seo"]
print("Canonicalization:")
for alias in test_aliases:
    canon = canonicalize(alias)
    changed = canon != alias
    print(f"  {'→' if changed else ' '} '{alias}' → '{canon}'")

Canonicalization:
  → 'TP HCM' → 'TP.HCM'
  → 'Facebook' → 'Meta'
  → 'Apple Inc.' → 'Apple'
  → 'Chat GPT' → 'ChatGPT'
  → 'Hà nội' → 'Hà Nội'
  → 'Park Hang-seo' → 'Park Hang Seo'


## Merge Engine + Tokenizer + BIO Converter + Canonicalization

Kết hợp 3 tầng, xử lý overlap, convert sang BIO format, chuẩn hoá tên entity.

**Priority:** Model > Regex > Gazetteer · **Overlap:** longest + highest priority wins

In [22]:
PRIORITY_MODEL = 3
PRIORITY_REGEX = 2
PRIORITY_GAZ   = 1


def resolve_overlaps(
    spans: List[Tuple[int, int, str, int]]
) -> List[Tuple[int, int, str]]:
    """
    Giải quyết overlap giữa các span.
    Input: list of (start, end, entity_type, priority)
    Output: list of (start, end, entity_type) không trùng lấn.

    Chiến lược:
      1. Priority cao hơn thắng
      2. Cùng priority → span dài hơn thắng
      3. Cùng length → span xuất hiện trước thắng
    """
    spans_sorted = sorted(spans, key=lambda x: (-x[3], -(x[1] - x[0]), x[0]))

    resolved = []
    occupied = set()
    for start, end, etype, _ in spans_sorted:
        char_range = set(range(start, end))
        if not char_range & occupied:
            resolved.append((start, end, etype))
            occupied |= char_range

    resolved.sort(key=lambda x: x[0])
    return resolved


def hybrid_ner(text: str) -> List[Tuple[int, int, str]]:
    """
    Pipeline NER hybrid: normalize → Model + Regex + Gazetteer → merge.
    Text được normalize trước khi chạy qua 3 layers.
    """
    text = normalize_text(text)
    if not text:
        return []

    all_spans = []

    for start, end, etype, score in electra_ner(text):
        all_spans.append((start, end, etype, PRIORITY_MODEL))

    for start, end, etype in regex_ner(text):
        all_spans.append((start, end, etype, PRIORITY_REGEX))

    for start, end, etype in gazetteer_ner(text):
        all_spans.append((start, end, etype, PRIORITY_GAZ))

    return resolve_overlaps(all_spans)


# === Tokenizer with offsets ===

def tokenize_with_offsets(text: str) -> List[Tuple[str, int, int]]:
    """
    Tách token theo khoảng trắng + tách dấu câu dính.
    Trả về (token, start_char, end_char).
    """
    if not isinstance(text, str) or not text.strip():
        return []

    result = []
    raw_tokens = text.split()
    pos = 0

    for raw in raw_tokens:
        idx = text.find(raw, pos)
        if idx == -1:
            idx = pos
        offset = idx

        # Leading punctuation
        i = 0
        while i < len(raw) and raw[i] in '([{"\'"«»\u201c\u201d\u2018\u2019':
            result.append((raw[i], offset + i, offset + i + 1))
            i += 1

        # Trailing punctuation
        j = len(raw)
        trailing = []
        while j > i and raw[j - 1] in '.,;:!?)]}"\'"»\u201d\u2019\u2026':
            if raw[j - 1] in '.,' and j - 2 >= i and raw[j - 2].isdigit():
                break
            trailing.append((raw[j - 1], offset + j - 1, offset + j))
            j -= 1

        if j > i:
            result.append((raw[i:j], offset + i, offset + j))

        for t in reversed(trailing):
            result.append(t)

        pos = idx + len(raw)

    return result


# === Spans → BIO labels ===

def spans_to_bio(
    text: str,
    entity_spans: List[Tuple[int, int, str]]
) -> List[Tuple[str, str]]:
    """
    Chuyển character-level entity spans thành token-level BIO labels.
    Trả về list of (token, label).
    """
    token_data = tokenize_with_offsets(text)
    result = []

    for token, t_start, t_end in token_data:
        label = "O"
        for e_start, e_end, e_type in entity_spans:
            if t_start >= e_start and t_end <= e_end:
                label = f"B-{e_type}" if t_start == e_start else f"I-{e_type}"
                break
        result.append((token, label))

    return result


def hybrid_ner_bio(text: str) -> List[Tuple[str, str]]:
    """Full pipeline: normalize → hybrid NER → BIO labels."""
    text = normalize_text(text)
    if not text:
        return []
    spans = hybrid_ner(text)
    return spans_to_bio(text, spans)


print("Merge engine ready.")

Merge engine ready.


## Test Hybrid Pipeline

So sánh kết quả từng layer và kết quả merged.

In [23]:
def show_ner_comparison(text: str):
    """Hiển thị kết quả từng layer + merged."""
    print(f"{'='*70}")
    print(f"INPUT: {text[:100]}{'...' if len(text)>100 else ''}")
    print(f"{'='*70}")

    # Layer 1: Model
    model_spans = electra_ner(text)
    print(f"\n[Layer 1 - Electra Model] ({len(model_spans)} entities)")
    for s, e, t, sc in model_spans:
        print(f"  {t:15s} ({sc:.2f}) │ '{text[s:e]}'")

    # Layer 2: Regex
    regex_spans = regex_ner(text)
    print(f"\n[Layer 2 - Regex] ({len(regex_spans)} entities)")
    for s, e, t in regex_spans:
        print(f"  {t:15s}        │ '{text[s:e]}'")

    # Layer 3: Gazetteer
    gaz_spans = gazetteer_ner(text)
    print(f"\n[Layer 3 - Gazetteer] ({len(gaz_spans)} entities)")
    for s, e, t in gaz_spans:
        print(f"  {t:15s}        │ '{text[s:e]}'")

    # Merged
    merged = hybrid_ner(text)
    print(f"\n[MERGED] ({len(merged)} entities)")
    for s, e, t in merged:
        print(f"  {t:15s}        │ '{text[s:e]}'")

    # BIO output
    bio = hybrid_ner_bio(text)
    print(f"\n[BIO Output]")
    for token, label in bio:
        if label != "O":
            print(f"  {token:25s} {label}")

    print()


# Test 1: Bài đầu tiên
show_ner_comparison(str(df.iloc[0]["title"]))

# Test 2: Câu phức tạp
show_ner_comparison(
    "Ngày 16/1, ông Roland Staub đến Bình Định, cam kết hỗ trợ 50 triệu USD "
    "cho ngành du lịch, tăng 6,5% so với năm 2024."
)

# Test 3: Công nghệ
show_ner_comparison(
    "Apple bắt tay Google: Gemini chính thức trở thành bộ não mới cho Siri, "
    "thỏa thuận trị giá 1 tỷ USD tại WWDC 2024."
)

INPUT: Motorola chuẩn bị ra mắt phiên bản điện thoại nắp gập Razr giá rẻ 12 triệu đồng

[Layer 1 - Electra Model] (1 entities)
  ORGANIZATION    (1.00) │ 'Motorola'

[Layer 2 - Regex] (2 entities)
  MONEY                  │ '12 triệu đồng'
  MONEY                  │ '12 triệu'

[Layer 3 - Gazetteer] (1 entities)
  PRODUCT                │ 'Razr'

[MERGED] (3 entities)
  ORGANIZATION           │ 'Motorola'
  PRODUCT                │ 'Razr'
  MONEY                  │ '12 triệu đồng'

[BIO Output]
  Motorola                  B-ORGANIZATION
  Razr                      B-PRODUCT
  12                        B-MONEY
  triệu                     I-MONEY
  đồng                      I-MONEY

INPUT: Ngày 16/1, ông Roland Staub đến Bình Định, cam kết hỗ trợ 50 triệu USD cho ngành du lịch, tăng 6,5% ...

[Layer 1 - Electra Model] (2 entities)
  PERSON          (1.00) │ 'Roland Staub'
  LOCATION        (1.00) │ 'Bình Định'

[Layer 2 - Regex] (6 entities)
  MONEY                  │ '50 triệu USD'
  MO

## Áp dụng lên toàn bộ dataset

Batch processing với progress tracking. Content dài được cắt chunk để model xử lý hiệu quả.

In [24]:
def bio_to_string(bio: List[Tuple[str, str]]) -> str:
    return "\n".join(f"{token} {label}" for token, label in bio)


def batch_electra_ner(
    texts: List[str], min_score: float = 0.70
) -> List[List[Tuple[int, int, str, float]]]:
    """
    Batch inference: chạy Electra NER trên nhiều text cùng lúc.
    GPU sẽ xử lý batch_size=16 text song song → nhanh hơn ~10x so với từng text.
    """
    all_results = [[] for _ in texts]
    chunk_map = []  # (text_idx, chunk_text, offset)

    for i, text in enumerate(texts):
        if not isinstance(text, str) or not text.strip():
            continue
        MAX_CHARS = 1500
        if len(text) <= MAX_CHARS:
            chunk_map.append((i, text, 0))
        else:
            sentences = re.split(r'(?<=[.!?])\s+', text)
            current_chunk, current_offset = "", 0
            for sent in sentences:
                if len(current_chunk) + len(sent) + 1 > MAX_CHARS and current_chunk:
                    chunk_map.append((i, current_chunk, current_offset))
                    current_offset += len(current_chunk) + 1
                    current_chunk = sent
                else:
                    current_chunk = (current_chunk + " " + sent) if current_chunk else sent
            if current_chunk:
                chunk_map.append((i, current_chunk, current_offset))

    if not chunk_map:
        return all_results

    chunk_texts = [c[1] for c in chunk_map]

    try:
        pipe_results = ner_pipeline(chunk_texts)
    except Exception:
        pipe_results = [[] for _ in chunk_texts]

    if chunk_texts and not isinstance(pipe_results[0], list):
        pipe_results = [pipe_results]

    for (text_idx, _, offset), ents in zip(chunk_map, pipe_results):
        for ent in ents:
            raw_label = ent["entity_group"]
            mapped = ELECTRA_LABEL_MAP.get(raw_label, raw_label)
            if ent["score"] < min_score or mapped == "MISCELLANEOUS":
                continue
            all_results[text_idx].append(
                (ent["start"] + offset, ent["end"] + offset, mapped, ent["score"])
            )

    return all_results


def process_column_gpu(series: pd.Series, col_name: str, batch_sz: int = 64) -> pd.Series:
    """
    Batch processing trên GPU.
    - Gom batch_sz texts → chạy Electra 1 lần cho cả batch
    - Regex + Gazetteer vẫn chạy per-text (rất nhanh, ko cần GPU)
    """
    total = len(series)
    texts = [normalize_text(str(t)) if pd.notna(t) else "" for t in series]
    results = [""] * total
    start_time = time.time()

    for batch_start in range(0, total, batch_sz):
        batch_end = min(batch_start + batch_sz, total)
        batch_texts = texts[batch_start:batch_end]

        # Layer 1: Electra batch inference on GPU
        model_results = batch_electra_ner(batch_texts)

        for j, text in enumerate(batch_texts):
            idx = batch_start + j
            all_spans = []

            # Model spans
            for s, e, t, sc in model_results[j]:
                all_spans.append((s, e, t, PRIORITY_MODEL))

            # Layer 2+3: Regex + Gazetteer (CPU, rất nhanh)
            for s, e, t in regex_ner(text):
                all_spans.append((s, e, t, PRIORITY_REGEX))
            for s, e, t in gazetteer_ner(text):
                all_spans.append((s, e, t, PRIORITY_GAZ))

            merged = resolve_overlaps(all_spans)
            bio = spans_to_bio(text, merged)
            results[idx] = bio_to_string(bio)

        done = batch_end
        elapsed = time.time() - start_time
        speed = done / elapsed if elapsed > 0 else 0
        eta = (total - done) / speed if speed > 0 else 0
        vram = torch.cuda.memory_allocated(0) / 1024**2

        if done % 500 < batch_sz or done == total:
            print(f"  [{col_name}] {done:>6,}/{total:,} "
                  f"({done/total*100:5.1f}%) "
                  f"| {elapsed:.0f}s | ~{eta:.0f}s left | "
                  f"{speed:.1f} rows/s | VRAM {vram:.0f}MB")

    return pd.Series(results, index=series.index)


# === Xử lý TITLE trên GPU ===
torch.cuda.empty_cache()
print("=" * 60)
print("PROCESSING TITLES (GPU batch inference)")
print("=" * 60)
t0 = time.time()
df["title_bio"] = process_column_gpu(df["title"], "title", batch_sz=64)
title_time = time.time() - t0
print(f"\nTitle done: {title_time:.1f}s\n")

PROCESSING TITLES (GPU batch inference)
  [title]    512/13,906 (  3.7%) | 1s | ~27s left | 500.8 rows/s | VRAM 1027MB
  [title]  1,024/13,906 (  7.4%) | 2s | ~26s left | 501.4 rows/s | VRAM 1027MB
  [title]  1,536/13,906 ( 11.0%) | 3s | ~24s left | 505.4 rows/s | VRAM 1027MB
  [title]  2,048/13,906 ( 14.7%) | 4s | ~24s left | 501.3 rows/s | VRAM 1027MB
  [title]  2,560/13,906 ( 18.4%) | 5s | ~23s left | 494.3 rows/s | VRAM 1027MB
  [title]  3,008/13,906 ( 21.6%) | 6s | ~22s left | 486.8 rows/s | VRAM 1027MB
  [title]  3,520/13,906 ( 25.3%) | 7s | ~22s left | 475.6 rows/s | VRAM 1027MB
  [title]  4,032/13,906 ( 29.0%) | 9s | ~21s left | 465.5 rows/s | VRAM 1027MB
  [title]  4,544/13,906 ( 32.7%) | 10s | ~20s left | 461.8 rows/s | VRAM 1027MB
  [title]  5,056/13,906 ( 36.4%) | 11s | ~19s left | 457.8 rows/s | VRAM 1027MB
  [title]  5,504/13,906 ( 39.6%) | 12s | ~18s left | 456.4 rows/s | VRAM 1027MB
  [title]  6,016/13,906 ( 43.3%) | 13s | ~17s left | 452.7 rows/s | VRAM 1027MB
  [title

In [25]:
# === Xử lý CONTENT trên GPU ===
# Content dài hơn → batch_sz nhỏ hơn để tránh OOM trên 8GB VRAM
torch.cuda.empty_cache()
print("=" * 60)
print("PROCESSING CONTENT (GPU batch inference)")
print("=" * 60)
t0 = time.time()
df["content_bio"] = process_column_gpu(df["content"], "content", batch_sz=16)
content_time = time.time() - t0

print(f"\nContent done: {content_time:.1f}s")
print(f"\n{'='*60}")
print(f"TỔNG THỜI GIAN: {title_time + content_time:.1f}s")
print(f"  Title:   {title_time:.1f}s")
print(f"  Content: {content_time:.1f}s")
print(f"  VRAM peak: {torch.cuda.max_memory_allocated(0)/1024**2:.0f} MB")

PROCESSING CONTENT (GPU batch inference)
  [content]    512/13,906 (  3.7%) | 49s | ~1293s left | 10.4 rows/s | VRAM 1027MB
  [content]  1,008/13,906 (  7.2%) | 94s | ~1207s left | 10.7 rows/s | VRAM 1027MB
  [content]  1,504/13,906 ( 10.8%) | 144s | ~1188s left | 10.4 rows/s | VRAM 1027MB
  [content]  2,000/13,906 ( 14.4%) | 191s | ~1134s left | 10.5 rows/s | VRAM 1027MB
  [content]  2,512/13,906 ( 18.1%) | 243s | ~1103s left | 10.3 rows/s | VRAM 1027MB
  [content]  3,008/13,906 ( 21.6%) | 292s | ~1058s left | 10.3 rows/s | VRAM 1027MB
  [content]  3,504/13,906 ( 25.2%) | 347s | ~1029s left | 10.1 rows/s | VRAM 1027MB
  [content]  4,000/13,906 ( 28.8%) | 401s | ~993s left | 10.0 rows/s | VRAM 1027MB
  [content]  4,512/13,906 ( 32.4%) | 456s | ~950s left | 9.9 rows/s | VRAM 1027MB
  [content]  5,008/13,906 ( 36.0%) | 509s | ~905s left | 9.8 rows/s | VRAM 1027MB
  [content]  5,504/13,906 ( 39.6%) | 558s | ~851s left | 9.9 rows/s | VRAM 1027MB
  [content]  6,000/13,906 ( 43.1%) | 609s | 

## Thống kê phân bố Entity

In [26]:
def count_entities(bio_col: pd.Series) -> Counter:
    """Đếm B-tag cho mỗi entity type."""
    counter = Counter()
    for bio_str in bio_col:
        if not isinstance(bio_str, str):
            continue
        for line in bio_str.split("\n"):
            parts = line.rsplit(" ", 1)
            if len(parts) == 2 and parts[1].startswith("B-"):
                counter[parts[1][2:]] += 1
    return counter


def count_all_labels(bio_col: pd.Series) -> Counter:
    """Đếm tất cả labels."""
    counter = Counter()
    for bio_str in bio_col:
        if not isinstance(bio_str, str):
            continue
        for line in bio_str.split("\n"):
            parts = line.rsplit(" ", 1)
            if len(parts) == 2:
                counter[parts[1]] += 1
    return counter


for col_name, bio_col in [("TITLE", df["title_bio"]), ("CONTENT", df["content_bio"])]:
    print(f"\n{'='*55}")
    print(f" ENTITY DISTRIBUTION — {col_name}")
    print(f"{'='*55}")

    entities = count_entities(bio_col)
    total_ent = sum(entities.values())

    for etype, cnt in entities.most_common():
        pct = cnt / total_ent * 100 if total_ent > 0 else 0
        bar = "█" * int(pct / 2)
        print(f"  {etype:15s} {cnt:>8,}  ({pct:5.1f}%) {bar}")

    print(f"  {'─'*45}")
    print(f"  {'TOTAL':15s} {total_ent:>8,}")

    # Token distribution
    labels = count_all_labels(bio_col)
    total_tokens = sum(labels.values())
    o_count = labels.get("O", 0)
    entity_tokens = total_tokens - o_count
    print(f"\n  Total tokens:  {total_tokens:>10,}")
    print(f"  O tokens:      {o_count:>10,}  ({o_count/total_tokens*100:.1f}%)")
    print(f"  Entity tokens: {entity_tokens:>10,}  ({entity_tokens/total_tokens*100:.1f}%)")


 ENTITY DISTRIBUTION — TITLE
  ORGANIZATION       4,936  ( 30.8%) ███████████████
  LOCATION           4,891  ( 30.5%) ███████████████
  MONEY              1,656  ( 10.3%) █████
  DATE               1,215  (  7.6%) ███
  PERSON               977  (  6.1%) ███
  PRODUCT              916  (  5.7%) ██
  INDUSTRY             906  (  5.7%) ██
  PERCENT              344  (  2.1%) █
  EVENT                181  (  1.1%) 
  ─────────────────────────────────────────────
  TOTAL             16,022

  Total tokens:     212,936
  O tokens:         182,440  (85.7%)
  Entity tokens:     30,496  (14.3%)

 ENTITY DISTRIBUTION — CONTENT
  ORGANIZATION     167,108  ( 32.2%) ████████████████
  LOCATION         111,261  ( 21.5%) ██████████
  DATE              53,894  ( 10.4%) █████
  PERSON            50,411  (  9.7%) ████
  MONEY             48,516  (  9.4%) ████
  PERCENT           35,711  (  6.9%) ███
  INDUSTRY          32,374  (  6.2%) ███
  PRODUCT           17,069  (  3.3%) █
  EVENT              2

## Post-processing: BIO Cleaning & Normalization

Làm sạch và chuẩn hóa nhãn BIO trước khi xuất file.

**Các rule được áp dụng:**
1. **Label consistency** — token đã biết luôn có đúng label type
2. **Entity type correction** — ChatGPT → PRODUCT, Pepero → PRODUCT, ...
3. **Force O** — `tỷ phú`, `triệu phú` không được label MONEY/PERSON
4. **MONEY tokenization** — `đồng/lít` → `đồng` (giữ label) + `/` O + `lít` O
5. **BIO validity** — `I-X` không có `B-X`/`I-X` trước → chuyển thành `B-X`
6. **Missing entities** — thêm label cho entity rõ ràng bị bỏ sót (MacBook, Galaxy S..., ...)

In [27]:
# ============================================================
# POST-PROCESSING: BIO Cleaning & Normalization
# Áp dụng trực tiếp lên df['title_bio'] và df['content_bio']
# trước khi export sang CoNLL / JSON / CSV.
# ============================================================

# ── 1. Single-token corrections ──────────────────────────────
# Format: token_text → (correct_entity_type, also_add_if_currently_O)
# also_add_if_currently_O=True: thêm label kể cả khi đang là O (entity rõ ràng bị bỏ sót)
# also_add_if_currently_O=False: chỉ sửa khi đang có label sai (tránh false positive)
_SINGLE_TOKEN_FIX: Dict[str, Tuple[str, bool]] = {
    # ── PRODUCT (unambiguous — always fix + add if missing) ──
    "ChatGPT"   : ("PRODUCT", True),
    "iPhone"    : ("PRODUCT", True),
    "iPad"      : ("PRODUCT", True),
    "MacBook"   : ("PRODUCT", True),
    "iMac"      : ("PRODUCT", True),
    "iPod"      : ("PRODUCT", True),
    "AirPods"   : ("PRODUCT", True),
    "HomePod"   : ("PRODUCT", True),
    "iOS"       : ("PRODUCT", True),
    "macOS"     : ("PRODUCT", True),
    "Siri"      : ("PRODUCT", True),
    "Copilot"   : ("PRODUCT", True),
    "Pepero"    : ("PRODUCT", True),
    "Razr"      : ("PRODUCT", True),
    "Snapdragon": ("PRODUCT", True),
    "MediaTek"  : ("PRODUCT", True),
    # ── PRODUCT (context-dependent — chỉ sửa nếu đang label sai) ──
    "Galaxy"    : ("PRODUCT", False),   # "Galaxy" alone: could be generic
    "Pixel"     : ("PRODUCT", False),   # "Pixel" alone: could be a unit
    "Gemini"    : ("PRODUCT", False),   # also a constellation
    "Alexa"     : ("PRODUCT", False),   # also a person name
    "Claude"    : ("PRODUCT", False),   # also a person name
    "DeepSeek"  : ("PRODUCT", False),
    "Llama"     : ("PRODUCT", False),
    # ── ORGANIZATION (chỉ sửa nếu label sai, không thêm nếu O) ──
    "OpenAI"    : ("ORGANIZATION", False),
    "Viettel"   : ("ORGANIZATION", False),
    "VIB"       : ("ORGANIZATION", False),
    "VPBank"    : ("ORGANIZATION", False),
    # ── EVENT (chỉ sửa nếu label sai) ────────────────────────
    "WWDC"      : ("EVENT", False),
    "Olympic"   : ("EVENT", False),
    "MWC"       : ("EVENT", False),
    "CES"       : ("EVENT", False),
}

# ── 2. Multi-token sequence corrections (sliding window) ─────
# Format: (token_tuple, correct_type_or_None, also_add_if_O)
# correct_type=None → force toàn bộ sequence về O
_MULTI_TOKEN_FIX: List[Tuple[tuple, Optional[str], bool]] = [
    # Force O: không label "tỷ phú" hay "triệu phú" là MONEY/PERSON
    (("tỷ",    "phú"),   None,         False),
    (("triệu", "phú"),   None,         False),
    # PRODUCT multi-token (add if missing)
    (("MacBook", "Pro"),  "PRODUCT",   True),
    (("MacBook", "Air"),  "PRODUCT",   True),
    (("Apple",   "Watch"),"PRODUCT",   True),
    (("Apple",   "TV"),   "PRODUCT",   True),
    (("Apple",   "M3"),   "PRODUCT",   True),
    (("Apple",   "M4"),   "PRODUCT",   True),
    (("Google",  "Pixel"),"PRODUCT",   True),
    # EVENT multi-token
    (("SEA",    "Games"), "EVENT",     True),
    (("Google", "I/O"),   "EVENT",     True),
]

# ── 3. Tokens to NOT split on "/" ─────────────────────────────
_NO_SPLIT_TOKENS: set = {
    "TP.HCM", "tp.hcm", "TP/HN", "tp/hn",
    "km/h", "m/s", "km/s", "USD/VND", "VND/USD",
    "R&D", "B2B", "B2C", "P2P",
}


# ─────────────────────────────────────────────────────────────
# Core helpers
# ─────────────────────────────────────────────────────────────

def _split_slash(token: str, label: str) -> List[Tuple[str, str]]:
    """
    Split tokens containing "/" (e.g. "đồng/lít" → "đồng", "/", "lít").
    First part keeps original label; "/" and rest become O.
    Preserves MONEY labeling for the currency word itself.
    """
    if token in _NO_SPLIT_TOKENS or "/" not in token:
        return [(token, label)]
    parts = re.split(r"(/)", token)
    parts = [p for p in parts if p]
    result = []
    first_done = False
    for part in parts:
        if part == "/":
            result.append(("/", "O"))
        elif not first_done:
            result.append((part, label))
            first_done = True
        else:
            result.append((part, "O"))
    return result


def _get_type(label: str) -> Optional[str]:
    if label.startswith(("B-", "I-")):
        return label[2:]
    return None


def _fix_bio_string(bio_str: str) -> str:
    """Apply all post-processing rules to one BIO-format string."""
    if not isinstance(bio_str, str) or not bio_str.strip():
        return bio_str

    # Parse into mutable list of [token, label]
    pairs: List[List[str]] = []
    for line in bio_str.strip().split("\n"):
        parts = line.rsplit(" ", 1)
        if len(parts) == 2 and parts[0].strip():
            pairs.append([parts[0], parts[1]])
        elif len(parts) == 1 and parts[0].strip():
            pairs.append([parts[0], "O"])
    if not pairs:
        return bio_str

    # ── Rule 4 & 5: Split "/" tokens ──────────────────────────
    expanded: List[List[str]] = []
    for tok, lbl in pairs:
        for new_tok, new_lbl in _split_slash(tok, lbl):
            expanded.append([new_tok, new_lbl])
    pairs = expanded

    # ── Rules 1, 2, 7: Single-token corrections ───────────────
    for i, (tok, lbl) in enumerate(pairs):
        if tok not in _SINGLE_TOKEN_FIX:
            continue
        correct_type, apply_if_o = _SINGLE_TOKEN_FIX[tok]
        if lbl == "O":
            if apply_if_o:
                pairs[i][1] = f"B-{correct_type}"
        elif lbl.startswith("B-") and lbl[2:] != correct_type:
            pairs[i][1] = f"B-{correct_type}"
        elif lbl.startswith("I-") and lbl[2:] != correct_type:
            pairs[i][1] = f"I-{correct_type}"

    # ── Rule 3: Multi-token corrections (sliding window) ──────
    n_pairs = len(pairs)
    for pattern, correct_type, apply_if_o in _MULTI_TOKEN_FIX:
        plen = len(pattern)
        for i in range(n_pairs - plen + 1):
            window_tokens = tuple(pairs[i + j][0] for j in range(plen))
            if window_tokens != pattern:
                continue
            window_labels = [pairs[i + j][1] for j in range(plen)]
            all_o = all(l == "O" for l in window_labels)
            any_wrong = any(
                _get_type(l) is not None and _get_type(l) != correct_type
                for l in window_labels
            )
            if correct_type is None:
                # Force to O (e.g., "tỷ phú", "triệu phú")
                for j in range(plen):
                    pairs[i + j][1] = "O"
            elif all_o and apply_if_o:
                # Add missing label
                pairs[i][1] = f"B-{correct_type}"
                for j in range(1, plen):
                    pairs[i + j][1] = f"I-{correct_type}"
            elif any_wrong:
                # Fix wrong entity type
                pairs[i][1] = f"B-{correct_type}"
                for j in range(1, plen):
                    pairs[i + j][1] = f"I-{correct_type}"

    # ── Rule 6: BIO validity — I-X without preceding B-X/I-X ──
    prev_type: Optional[str] = None
    for i, (tok, lbl) in enumerate(pairs):
        if lbl.startswith("I-"):
            ct = lbl[2:]
            if prev_type != ct:
                pairs[i][1] = f"B-{ct}"   # orphan I- → B-
                prev_type = ct
            else:
                prev_type = ct
        elif lbl.startswith("B-"):
            prev_type = lbl[2:]
        else:
            prev_type = None

    return "\n".join(f"{t} {l}" for t, l in pairs)


# ─────────────────────────────────────────────────────────────
# Apply to DataFrame + report stats
# ─────────────────────────────────────────────────────────────

def _count_labels(bio_col: pd.Series) -> Counter:
    c: Counter = Counter()
    for bio_str in bio_col:
        if not isinstance(bio_str, str):
            continue
        for line in bio_str.strip().split("\n"):
            parts = line.rsplit(" ", 1)
            if len(parts) == 2:
                c[parts[1]] += 1
    return c

print("=" * 60)
print("POST-PROCESSING: BIO Cleaning & Normalization")
print("=" * 60)

before_title   = _count_labels(df["title_bio"])
before_content = _count_labels(df["content_bio"])

df["title_bio"]   = df["title_bio"].apply(_fix_bio_string)
df["content_bio"] = df["content_bio"].apply(_fix_bio_string)

after_title   = _count_labels(df["title_bio"])
after_content = _count_labels(df["content_bio"])

# Merge before/after for diff report
all_labels_seen = sorted(
    set(before_title) | set(before_content) | set(after_title) | set(after_content)
)
print(f"\n{'Label':<22} {'Before':>10} {'After':>10} {'Delta':>8}")
print("-" * 54)
total_before = total_after = 0
for lbl in all_labels_seen:
    if lbl == "O":
        continue
    b = (before_title.get(lbl, 0) + before_content.get(lbl, 0))
    a = (after_title.get(lbl, 0)  + after_content.get(lbl, 0))
    delta = a - b
    total_before += b
    total_after  += a
    if delta != 0 or b > 0:
        marker = " ▲" if delta > 0 else (" ▼" if delta < 0 else "")
        print(f"  {lbl:<20} {b:>10,} {a:>10,} {delta:>+8,}{marker}")
print("-" * 54)
print(f"  {'TOTAL (non-O)':<20} {total_before:>10,} {total_after:>10,} {total_after - total_before:>+8,}")
print("\nPost-processing complete — ready for export.")

POST-PROCESSING: BIO Cleaning & Normalization

Label                      Before      After    Delta
------------------------------------------------------
  B-DATE                   55,109     55,115       +6 ▲
  B-EVENT                   2,269      2,287      +18 ▲
  B-INDUSTRY               33,280     33,289       +9 ▲
  B-LOCATION              116,152    116,292     +140 ▲
  B-MONEY                  50,172     56,324   +6,152 ▲
  B-ORGANIZATION          172,044    171,624     -420 ▼
  B-PERCENT                36,055     36,082      +27 ▲
  B-PERSON                 51,388     51,368      -20 ▼
  B-PRODUCT                17,985     26,368   +8,383 ▲
  I-DATE                   30,011     30,005       -6 ▼
  I-EVENT                   5,159      5,151       -8 ▼
  I-INDUSTRY               54,285     54,276       -9 ▼
  I-LOCATION              107,292    107,141     -151 ▼
  I-MONEY                  67,050     60,837   -6,213 ▼
  I-ORGANIZATION          224,669    224,489     -180 ▼
  I-

## Xuất dữ liệu NER

In [28]:
# === FORMAT 1: CoNLL ===

def write_conll(bio_col: pd.Series, path: str, append: bool = False) -> int:
    count = 0
    mode = "a" if append else "w"
    with open(path, mode, encoding="utf-8") as f:
        for bio_str in bio_col:
            if not isinstance(bio_str, str) or not bio_str.strip():
                continue
            for line in bio_str.strip().split("\n"):
                parts = line.rsplit(" ", 1)
                if len(parts) == 2:
                    f.write(f"{parts[0]}\t{parts[1]}\n")
            f.write("\n")
            count += 1
    return count

n1 = write_conll(df["title_bio"], "ner_conll4.txt")
n2 = write_conll(df["content_bio"], "ner_conll4.txt", append=True)
print(f"CoNLL: ner_conll2.txt ({n1 + n2:,} sentences — {n1:,} title + {n2:,} content)")

CoNLL: ner_conll2.txt (27,812 sentences — 13,906 title + 13,906 content)


In [29]:
# === FORMAT 2: JSON (HuggingFace-compatible + canonical entities) ===

def bio_str_to_record(row_id, text, bio_str, source_field="title"):
    tokens, labels = [], []
    if isinstance(bio_str, str) and bio_str.strip():
        for line in bio_str.strip().split("\n"):
            parts = line.rsplit(" ", 1)
            if len(parts) == 2:
                tokens.append(parts[0])
                labels.append(parts[1])
    return {
        "id":           int(row_id),
        "source_field": source_field,
        "text":         str(text),
        "tokens":       tokens,
        "ner_tags":     labels,
    }


def extract_canonical_entities(bio_str: str) -> List[Dict]:
    """Trích entities + canonical name từ BIO string."""
    entities = []
    buf_tokens, buf_type = [], None
    for line in str(bio_str).strip().split("\n"):
        parts = line.rsplit(" ", 1)
        if len(parts) != 2:
            continue
        token, label = parts
        if label.startswith("B-"):
            if buf_tokens and buf_type:
                surface = " ".join(buf_tokens)
                entities.append({"text": surface, "canonical": canonicalize(surface), "type": buf_type})
            buf_tokens, buf_type = [token], label[2:]
        elif label.startswith("I-") and buf_type:
            buf_tokens.append(token)
        else:
            if buf_tokens and buf_type:
                surface = " ".join(buf_tokens)
                entities.append({"text": surface, "canonical": canonicalize(surface), "type": buf_type})
            buf_tokens, buf_type = [], None
    if buf_tokens and buf_type:
        surface = " ".join(buf_tokens)
        entities.append({"text": surface, "canonical": canonicalize(surface), "type": buf_type})
    return entities


records = []
for _, r in df.iterrows():
    for field, bio_col in [("title", "title_bio"), ("content", "content_bio")]:
        rec = bio_str_to_record(r["id"], r[field], r[bio_col], field)
        rec["entities"] = extract_canonical_entities(r[bio_col])
        records.append(rec)

with open("ner_labeled4.json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)
print(f"JSON: ner_labeled4.json ({len(records):,} records — title + content)")

# Preview
print("\nSample entities (first title):")
for ent in records[0]["entities"]:
    marker = f" → '{ent['canonical']}'" if ent["canonical"] != ent["text"] else ""
    print(f"  [{ent['type']:15s}] '{ent['text']}'{marker}")

JSON: ner_labeled4.json (27,812 records — title + content)

Sample entities (first title):
  [ORGANIZATION   ] 'Motorola'
  [PRODUCT        ] 'Razr'
  [MONEY          ] '12 triệu đồng'


In [30]:
# === FORMAT 3: CSV ===

df.to_csv("ner_labeled_data4.csv", index=False, encoding="utf-8-sig")
print(f"CSV: ner_labeled_data4.csv ({len(df):,} rows, {list(df.columns)})")

CSV: ner_labeled_data4.csv (13,906 rows, ['id', 'title', 'content', 'source', 'date', 'title_bio', 'content_bio'])


## Quality Check

Kiểm tra mẫu ngẫu nhiên + thống kê coverage.

In [31]:
import random
random.seed(42)

def extract_entities_from_bio(bio_str: str) -> List[Tuple[str, str]]:
    """Trích entity text từ BIO string."""
    entities = []
    current_tokens = []
    current_type = None

    for line in str(bio_str).strip().split("\n"):
        parts = line.rsplit(" ", 1)
        if len(parts) != 2:
            continue
        token, label = parts

        if label.startswith("B-"):
            if current_tokens and current_type:
                entities.append((" ".join(current_tokens), current_type))
            current_tokens = [token]
            current_type = label[2:]
        elif label.startswith("I-") and current_type:
            current_tokens.append(token)
        else:
            if current_tokens and current_type:
                entities.append((" ".join(current_tokens), current_type))
            current_tokens = []
            current_type = None

    if current_tokens and current_type:
        entities.append((" ".join(current_tokens), current_type))

    return entities


# Random sample check
sample_ids = random.sample(range(len(df)), min(8, len(df)))

for idx in sample_ids:
    row = df.iloc[idx]
    title = str(row["title"])

    print(f"{'─'*60}")
    print(f"ID {row['id']}: {title[:75]}{'...' if len(title)>75 else ''}")

    entities = extract_entities_from_bio(row["title_bio"])
    if entities:
        for text, etype in entities:
            print(f"  ● {etype:15s} → {text}")
    else:
        print("  (no entities)")

print(f"\n{'═'*60}")
print("COVERAGE STATS")
print(f"{'═'*60}")

no_entity = df["title_bio"].apply(
    lambda x: not any(
        line.rsplit(" ", 1)[-1].startswith("B-")
        for line in str(x).strip().split("\n")
        if line.strip()
    )
)
n_empty = no_entity.sum()
print(f"Titles without any entity: {n_empty:,} / {len(df):,} ({n_empty/len(df)*100:.1f}%)")
print(f"Titles with ≥1 entity:    {len(df)-n_empty:,} / {len(df):,} ({(len(df)-n_empty)/len(df)*100:.1f}%)")

if n_empty > 0:
    print(f"\nSample titles without entities:")
    for _, r in df[no_entity].head(5).iterrows():
        print(f"  ID {r['id']}: {str(r['title'])[:75]}")

────────────────────────────────────────────────────────────
ID 10477: 'Cơn sốt' mở cửa hàng Mixue ở Việt Nam hạ nhiệt, la liệt tin rao nhượng lại...
  ● LOCATION        → Việt Nam
────────────────────────────────────────────────────────────
ID 1825: Gia hạn nhận hồ sơ tham gia Giải thưởng Chuyển đổi số Việt Nam 2025
  ● EVENT           → Giải thưởng Chuyển đổi số Việt Nam 2025
────────────────────────────────────────────────────────────
ID 410: Diễn biến mới của cổ phiếu Hóa chất Đức Giang trong phiên VN-Index tăng 47 ...
  (no entities)
────────────────────────────────────────────────────────────
ID 12150: Mô hình AI mới cho phép tấn công máy tính từ xa thông qua bức xạ điện từ
  (no entities)
────────────────────────────────────────────────────────────
ID 4507: Giá vàng tăng nóng, 'cá mập' còn mạnh tay gom vàng?
  (no entities)
────────────────────────────────────────────────────────────
ID 4013: Biến động kinh tế, gửi niềm tin vào vàng, trái phiếu hay USD?
  (no entities)
─────────

## Xem chi tiết BIO output mẫu

In [ ]:
# Hiển thị full BIO output cho 3 bài đầu
for i in range(min(3, len(df))):
    row = df.iloc[i]
    title = str(row["title"])
    bio_str = row["title_bio"]

    print(f"{'═'*60}")
    print(f"BÀI {row['id']}: {title[:70]}...")
    print(f"{'═'*60}")

    if isinstance(bio_str, str) and bio_str.strip():
        for line in bio_str.split("\n"):
            parts = line.rsplit(" ", 1)
            if len(parts) == 2:
                token, label = parts
                marker = " ◀" if label != "O" else ""
                print(f"  {token:30s} {label}{marker}")
    print()

════════════════════════════════════════════════════════════
BÀI 1: Motorola chuẩn bị ra mắt phiên bản điện thoại nắp gập Razr giá rẻ 12 t...
════════════════════════════════════════════════════════════
  Motorola                       B-ORGANIZATION ◀
  chuẩn                          O
  bị                             O
  ra                             O
  mắt                            O
  phiên                          O
  bản                            O
  điện                           O
  thoại                          O
  nắp                            O
  gập                            O
  Razr                           B-PRODUCT ◀
  giá                            O
  rẻ                             O
  12                             B-MONEY ◀
  triệu                          I-MONEY ◀
  đồng                           I-MONEY ◀

════════════════════════════════════════════════════════════
BÀI 2: 2 tỷ phú quốc tế 'xông đất', hứa đưa Bình Định thành trung tâm du lịch...
══════════

## Tổng kết

### Kiến trúc Hybrid NER Pipeline (GPU-accelerated)

| Layer | Nguồn | Entity Types | Cải tiến |
|-------|-------|-------------|---------|
| **0. Normalization** | `normalize_text()` | — | Unicode NFC, NBSP/ZWS, smart quotes |
| **1. Electra Model** | `NlpHUST/ner-vietnamese-electra-base` | PERSON, ORG, LOC | Batch GPU, F1=0.92 |
| **2. Regex** | Compiled patterns | MONEY, DATE, PERCENT, **PRODUCT** | Boundary-safe · PRODUCT family patterns |
| **3. Gazetteer** | Aho-Corasick | PRODUCT (HW+SW), EVENT, INDUSTRY | O(n) · contextual INDUSTRY filter |
| **4. Canonicalize** | `CANONICAL_MAP` | — | Alias → canonical (JSON metadata) |

### Gazetteer Improvements
- **Aho-Corasick:** O(n) matching — ~20x nhanh hơn str.find loop
- **HARDWARE / SOFTWARE split:** chip/device vs app/AI model
- **Contextual INDUSTRY:** "game", "công nghệ"... chỉ tag khi có trigger word ("ngành", "lĩnh vực"...) trong window ±50 chars → giảm false positive
- **PRODUCT regex families:** iPhone/Galaxy/Pixel/iOS/Android/Snapdragon tự cover model mới

### GPU Acceleration
- **Model:** batch inference trên RTX 3070 8GB
- **Title:** batch_size=64 · **Content:** batch_size=16

### Merge Strategy
- **Priority:** Model(3) > Regex(2) > Gazetteer(1)
- **Overlap:** highest priority thắng · cùng priority → span dài hơn thắng

### Output Files
| File | Format | Nội dung |
|------|--------|---------|
| `ner_conll.txt` | CoNLL | token\\tBIO (title + content) |
| `ner_labeled.json` | JSON | tokens + ner_tags + **entities + canonical** |
| `ner_labeled_data.csv` | CSV | Full DataFrame + title_bio + content_bio |